# Wan 2.2 Image-to-Video — Official Repository (T4 Colab)

Rebuilt from scratch on the **official [`Wan-Video/Wan2.2`](https://github.com/Wan-Video/Wan2.2) GitHub repository** — native inference (`generate.py`), not the `diffusers` integration. `AutoPipelineForImage2Video` is never imported anywhere in this notebook, so it cannot raise that error.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

**Model:** `Wan-AI/Wan2.2-TI2V-5B`, run via the official `ti2v-5B` task — the only Wan 2.2 checkpoint the repo positions for a single consumer-class GPU. The 14B `T2V-A14B` / `I2V-A14B` models are not used here: the official README states those need **at least 80GB VRAM** on a single GPU.

**Honest T4 limitation:** the official README states TI2V-5B itself needs **at least 24GB VRAM**, even with every memory-saving flag this notebook enables (`--offload_model True --convert_model_dtype --t5_cpu`). A free Colab T4 has 16GB. This notebook still targets a T4 with the smallest supported 9:16 size (`704*1280`, verified against the repo's own `SUPPORTED_SIZES`) and the model's own tested default length (121 frames ≈ 5s @ 24fps) — the best-effort configuration most likely to fit — but it may still hit a CUDA out-of-memory error depending on what Colab allocates that session. `DURATION_SECONDS` is a plain variable in the generation cell if you want to try longer, or need to shrink it further. If it still OOMs on your session, the reliable fix is more VRAM (Colab Pro's A100/L4), not a different flag — this notebook already sets every official low-VRAM option.

**`flash_attn` is intentionally skipped.** It's in the repo's `requirements.txt`, but its own attention module (`wan/modules/attention.py`) falls back to PyTorch's native `scaled_dot_product_attention` when `flash_attn` isn't installed — confirmed by reading that file directly. Compiling `flash_attn` in Colab is slow and failure-prone (a known issue the repo's own `INSTALL.md` calls out); skipping it trades a little speed for a install step that actually finishes.

## 0. Confirm the GPU

In [ ]:
!nvidia-smi

## 1. Clone the official Wan 2.2 repository

In [ ]:
%cd /content
!git clone https://github.com/Wan-Video/Wan2.2.git
%cd /content/Wan2.2

## 2. Install only the required packages

Colab's preinstalled `torch` is already CUDA-matched to the T4 runtime, so it's left alone rather than reinstalled (reinstalling `torch` in Colab is a common way to accidentally break GPU support). `flash_attn` is filtered out for the reason above.

In [ ]:
!grep -viE '^(flash_attn|torch|torchvision|torchaudio)' requirements.txt > requirements_colab.txt
!pip install -q -r requirements_colab.txt
!pip install -q "huggingface_hub[cli]"

import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU detected — set Runtime > Change runtime type > T4 GPU, then re-run.'
assert tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) >= (2, 4), \
    f'Wan 2.2 needs torch>=2.4.0, Colab has {torch.__version__}. Runtime > Disconnect and delete runtime, then re-run from the top.'

## 3. Download the T4-compatible Wan 2.2 model (TI2V-5B)

Several GB from Hugging Face — first run takes a few minutes.

In [ ]:
!huggingface-cli download Wan-AI/Wan2.2-TI2V-5B --local-dir ./Wan2.2-TI2V-5B

## 4. Upload your product image

In [ ]:
from google.colab import files

print('Choose one product photo to upload:')
uploaded = files.upload()  # saves into the current directory: /content/Wan2.2
image_filename = next(iter(uploaded))
print(f'Uploaded: {image_filename}')

## 5. Generate a 9:16 vertical video (official inference script)

`704*1280` is one of the two sizes the repo's own `SUPPORTED_SIZES['ti2v-5B']` actually allows (the other, `1280*704`, is landscape) — the nearest official equivalent to 9:16. `DURATION_SECONDS` defaults to the model's own tested length; raise it if you have VRAM headroom, lower it first if you hit an out-of-memory error. Frame counts must be `4n+1`, handled automatically below.

In [ ]:
import subprocess

# --- Tunables ---------------------------------------------------------
SIZE = '704*1280'          # official 9:16 portrait size for the ti2v-5B task
FPS = 24                   # fixed by the ti2v-5B model config (sample_fps)
DURATION_SECONDS = 5       # ~121 frames — the model's own tested default; raise with caution on a T4
PROMPT = 'A realistic, premium commercial product video. Natural, smooth camera motion, cinematic lighting.'  # edit me
SAVE_FILE = '/content/Wan2.2/output.mp4'
# ------------------------------------------------------------------------

FRAME_NUM = int(round((FPS * DURATION_SECONDS - 1) / 4)) * 4 + 1  # must be 4n+1
IMAGE_PATH = f'/content/Wan2.2/{image_filename}'

print(f'Requesting {FRAME_NUM} frames (~{FRAME_NUM / FPS:.1f}s @ {FPS}fps) at {SIZE}.')

result = subprocess.run(
    [
        'python', 'generate.py',
        '--task', 'ti2v-5B',
        '--size', SIZE,
        '--ckpt_dir', './Wan2.2-TI2V-5B',
        '--offload_model', 'True',
        '--convert_model_dtype',
        '--t5_cpu',
        '--image', IMAGE_PATH,
        '--prompt', PROMPT,
        '--frame_num', str(FRAME_NUM),
        '--save_file', SAVE_FILE,
    ],
    check=True,
)
print('Done:', SAVE_FILE)

## 6. Preview and auto-download the MP4

In [ ]:
from IPython.display import Video, display
display(Video(SAVE_FILE, embed=True))

files.download(SAVE_FILE)

## 7. (Optional) Expose a temporary API via Cloudflare Tunnel

For wiring this up to **BNK AI Ad Studio** later. Loads the model **once** into memory (unlike step 5, which spins up a fresh process per run) and serves it over Flask, then tunnels it publicly with `cloudflared` — Cloudflare's official tunnel client, downloaded directly since it isn't a pip package. This section is optional: skip it if you only needed the single video from steps 1–6.

The public URL is intentionally **temporary** — it only works while this Colab runtime stays connected.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
import sys
sys.path.insert(0, '/content/Wan2.2')

import wan
from wan.configs import WAN_CONFIGS, SIZE_CONFIGS, MAX_AREA_CONFIGS

API_CFG = WAN_CONFIGS['ti2v-5B']

print('Loading the WanTI2V pipeline once for the API (reuses the weights already downloaded)...')
wan_ti2v = wan.WanTI2V(
    config=API_CFG,
    checkpoint_dir='./Wan2.2-TI2V-5B',
    device_id=0,
    rank=0,
    t5_cpu=True,
    convert_model_dtype=True,
)
print('Pipeline ready.')

In [ ]:
from flask import Flask, request, send_file, jsonify
from threading import Thread
from wan.utils.utils import save_video
from PIL import Image
import tempfile, os, uuid, io, random, requests as pyrequests

api = Flask(__name__)

@api.get('/health')
def health():
    return jsonify({'status': 'ok'})

@api.post('/generate')
def api_generate():
    prompt = request.form.get('prompt', PROMPT)
    duration = float(request.form.get('duration', DURATION_SECONDS))
    frame_num = int(round((FPS * duration - 1) / 4)) * 4 + 1

    img = None
    if 'image' in request.files:
        img = Image.open(request.files['image'].stream).convert('RGB')
    elif request.form.get('image_url'):
        resp = pyrequests.get(request.form['image_url'], timeout=30)
        resp.raise_for_status()
        img = Image.open(io.BytesIO(resp.content)).convert('RGB')
    else:
        return jsonify({'error': "Send a multipart 'image' file or an 'image_url' field."}), 400

    seed = int(request.form.get('seed', random.randint(0, 2**31 - 1)))

    video_tensor = wan_ti2v.generate(
        prompt,
        img=img,
        size=SIZE_CONFIGS[SIZE],
        max_area=MAX_AREA_CONFIGS[SIZE],
        frame_num=frame_num,
        shift=API_CFG.sample_shift,
        sample_solver='unipc',
        sampling_steps=API_CFG.sample_steps,
        guide_scale=API_CFG.sample_guide_scale,
        seed=seed,
        offload_model=True,
    )

    out_path = os.path.join(tempfile.gettempdir(), f'{uuid.uuid4().hex}.mp4')
    save_video(
        tensor=video_tensor[None],
        save_file=out_path,
        fps=API_CFG.sample_fps,
        nrow=1,
        normalize=True,
        value_range=(-1, 1),
    )

    return send_file(out_path, mimetype='video/mp4', as_attachment=True, download_name='video.mp4')

def run_api():
    api.run(host='0.0.0.0', port=5000)

Thread(target=run_api, daemon=True).start()
print('API running on http://localhost:5000  (GET /health, POST /generate)')

In [ ]:
import subprocess, re

tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:5000', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None
print('Starting Cloudflare Tunnel...')
for line in tunnel_process.stdout:
    print(line, end='')
    match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print(f'\nTemporary public API: {public_url}')
    print(f"Try it:  curl -F image=@your_photo.jpg -F prompt='...' {public_url}/generate -o out.mp4")
else:
    print('Could not detect the tunnel URL — check the log above for errors.')

### Notes

- The API and its public URL live only as long as this notebook's runtime is connected — reconnecting or re-running gets you a **new** `trycloudflare.com` URL each time, which is the intended, temporary behavior.
- To call this from BNK AI Ad Studio: `POST` multipart form data to `<public_url>/generate` with an `image` file (or an `image_url` field) and a `prompt` field; the response body is the raw `video/mp4`.
- If the official repo's `generate.py` CLI flags or `wan.WanTI2V` API change upstream, re-check `https://github.com/Wan-Video/Wan2.2` — steps 5 and 7 are the only cells that call into it directly.